07/11/2026 Este notebook tiene la responsabilidad de realizar la ingesta de los datos de recaudos ubicadod en /raw e insertarlos en la capa bronze

In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_pysmartdata")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "pysmartdata2026")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/2.RecaudosAcueducto_202607en adelante.csv"

In [0]:
df_recaudos = spark.read.option('header', True)\
                        .option('sep', ';')\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
recaudos_schema = StructType(fields=[StructField("suscriptor", IntegerType(), True),                               StructField("periodo", IntegerType(), True),
                                     
                                     StructField("entidad", IntegerType(), True),
                                     StructField("descripcion", StringType(), True),
                                     StructField("codigo_punto_recaudo", IntegerType(), True),
                                     StructField("descripcion_punto_recaudo", StringType(), True),
                                     # El archivo trae CAJA y su DESCRIPCION entre el punto de recaudo y la fecha de pago (13 columnas)
                                     StructField("codigo_caja", IntegerType(), True),
                                     StructField("descripcion_caja", StringType(), True),
                                     # FECPAGO viene como dd-MMM-yyyy con meses en español (ej. 12-AGO-2026): se lee como texto
                                     StructField("fecha_pago", StringType(), True),
                                     StructField("codigo_tarifa", IntegerType(), True),
                                     StructField("codigo_concepto", IntegerType(), True),
                                     StructField("concepto", StringType(), True),
                                     StructField("valor_pagado", StringType(), True),
                                     # Los valores monetarios vienen con coma decimal (ej. 7785,96): se leen como texto y se convierten a double en la celda de selección
])

In [0]:
df_recaudos_final = spark.read\
.option('header', True)\
.option('sep', ';')\
.schema(recaudos_schema)\
.csv(ruta)

In [0]:
# Convertir meses en español (ENE..DIC) a número para parsear la fecha de pago
meses_es = {"ENE": "01", "FEB": "02", "MAR": "03", "ABR": "04", "MAY": "05", "JUN": "06",
            "JUL": "07", "AGO": "08", "SEP": "09", "OCT": "10", "NOV": "11", "DIC": "12"}
mapa_meses = create_map(*[lit(x) for kv in meses_es.items() for x in kv])
fecha_pago_col = to_date(
    concat_ws("-",
              split(upper(trim(col("fecha_pago"))), "-").getItem(0),
              mapa_meses[split(upper(trim(col("fecha_pago"))), "-").getItem(1)],
              split(upper(trim(col("fecha_pago"))), "-").getItem(2)),
    "dd-MM-yyyy")

recaudos_selected_df = df_recaudos_final.select(col("suscriptor"), 
                                                col("periodo"), 
                                                col("entidad"), 
                                                col("descripcion"), 
                                                col("codigo_punto_recaudo"), 
                                                col("descripcion_punto_recaudo"), 
                                                fecha_pago_col.alias("fecha_pago"), 
                                                col("codigo_tarifa"), 
                                                col("codigo_concepto"), 
                                                col("concepto"), 
                                                regexp_replace(col("valor_pagado"), ",", ".").cast("double").alias("valor_pagado"))


In [0]:
recaudos_renamed_df = recaudos_selected_df.withColumnRenamed("suscriptor", "suscriptor") \
                                         .withColumnRenamed("periodo", "periodo") \
                                         .withColumnRenamed("entidad", "entidad") \
                                         .withColumnRenamed("descripcion", "descripcion") \
                                         .withColumnRenamed("codigo_punto_recaudo", "codigo_punto_recaudo") \
                                          .withColumnRenamed("descripcion_punto_recaudo", "descripcion_punto_recaudo") \
                                          .withColumnRenamed("fecha_pago", "fecha_pago") \
                                           .withColumnRenamed("codigo_tarifa", "codigo_tarifa") \
                                           .withColumnRenamed("codigo_concepto", "codigo_concepto") \
                                           .withColumnRenamed("concepto", "concepto") \
                                           .withColumnRenamed("valor_pagado", "valor_pagado") \
                                            .withColumnRenamed("ingestion_date", "ingestion_date")
 

In [0]:
recaudos_final_df = recaudos_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
recaudos_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.recaudos")

In [0]:
tabla = f"{catalogo}.{esquema}.recaudos"
df_tabla = spark.table(tabla)

# 1. Comparar conteo de filas: origen (CSV) vs tabla destino
filas_origen = recaudos_final_df.count()
filas_tabla = df_tabla.count()
print(f"Filas origen: {filas_origen} | Filas en tabla: {filas_tabla} | Coinciden: {filas_origen == filas_tabla}")

# 2. Resumen por periodo, nulos en llaves y fecha de ingesta
display(
    df_tabla.groupBy("periodo").agg(
        count("*").alias("filas"),
        countDistinct("suscriptor").alias("suscriptores"),
        sum(when(col("suscriptor").isNull(), 1).otherwise(0)).alias("suscriptor_nulo"),
        round(sum("valor_pagado"), 2).alias("valor_pagado"),
        max("ingestion_date").alias("ultima_ingesta")
    ).orderBy("periodo")
)

# 3. Muestra de registros
display(df_tabla.limit(10))